# MedCLIP — NIH Zero-shot Evaluation
**Datasets required:**
- `maxzhang646/medclip-checkpoint`
- `nih-chest-xrays/data`

In [ ]:
!pip install -q ftfy regex
!pip install -q git+https://github.com/openai/CLIP.git --no-deps
!pip install -q transformers scikit-learn
!git clone -q https://github.com/maxzhang646/medical-clip.git
import sys
sys.path.insert(0, 'medical-clip/src')

In [ ]:
import os, glob
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from transformers import AutoTokenizer
from sklearn.metrics import roc_auc_score
from model import MedCLIP
from dataset import NIHDataset
from prompts import build_prompts

# Force CPU to avoid P100 CUDA compatibility issue
device = torch.device('cpu')
print(f'Device: {device}')

In [ ]:
# Auto-detect paths
ckpt_candidates = glob.glob('/kaggle/input/**/best.pt', recursive=True)
assert ckpt_candidates, 'best.pt not found!'
CKPT_PATH = ckpt_candidates[0]

nih_candidates = glob.glob('/kaggle/input/**/Data_Entry_2017.csv', recursive=True)
assert nih_candidates, 'NIH not found!'
NIH_DIR = os.path.dirname(nih_candidates[0])

print(f'CKPT_PATH: {CKPT_PATH}')
print(f'NIH_DIR:   {NIH_DIR}')

In [ ]:
# Load model
tokenizer = AutoTokenizer.from_pretrained('medicalai/ClinicalBERT')
model = MedCLIP().to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device, weights_only=False))
model.eval()
print('Model loaded.')

In [ ]:
DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Effusion', 'Infiltration', 'Pneumonia', 'Pneumothorax'
]

# Use 2000-sample subset for speed on CPU
full_ds = NIHDataset(NIH_DIR, classes=DISEASES)
np.random.seed(42)
idx = np.random.choice(len(full_ds), size=min(2000, len(full_ds)), replace=False)
nih_ds = Subset(full_ds, idx)
nih_loader = DataLoader(nih_ds, batch_size=64, shuffle=False, num_workers=2)
print(f'NIH eval samples: {len(nih_ds)}')

In [ ]:
@torch.no_grad()
def encode_prompts(model, tokenizer, diseases, prompt_key):
    embs = []
    for d in diseases:
        prompts = build_prompts(d)[prompt_key]
        enc = tokenizer(prompts, padding=True, truncation=True,
                        max_length=128, return_tensors='pt')
        e = model.encode_text(enc['input_ids'], enc['attention_mask'])
        embs.append(e.mean(dim=0))
    return torch.stack(embs)

@torch.no_grad()
def run_zeroshot(model, tokenizer, loader, diseases, prompt_key):
    text_embs = encode_prompts(model, tokenizer, diseases, prompt_key)
    all_logits, all_labels = [], []
    for batch in loader:
        img_emb = model.encode_image(batch['image'])
        logits  = img_emb @ text_embs.T
        all_logits.append(logits.numpy())
        all_labels.append(batch['labels'].numpy())
    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    aucs = {}
    for i, cls in enumerate(diseases):
        if labels[:, i].sum() > 0:
            aucs[cls] = roc_auc_score(labels[:, i], logits[:, i])
    aucs['Macro AUC'] = np.mean(list(aucs.values()))
    return aucs

PROMPT_KEYS = ['simple', 'findings', 'clinical', 'patient', 'radiologist', 'ensemble']
all_results = {}
for pk in PROMPT_KEYS:
    aucs = run_zeroshot(model, tokenizer, nih_loader, DISEASES, pk)
    all_results[pk] = aucs
    print(f'[{pk:12s}] Macro AUC: {aucs["Macro AUC"]:.4f}')

In [ ]:
# Full table
df = pd.DataFrame(all_results).T
print('\n=== Zero-shot AUC by prompt ===')
print(df.round(4).to_string())

In [ ]:
# Bar chart
macro_aucs = {k: all_results[k]['Macro AUC'] for k in PROMPT_KEYS}
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(macro_aucs.keys(), macro_aucs.values(), color='steelblue', width=0.6)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.set_ylim(0.4, 0.8)
ax.set_ylabel('Macro AUC')
ax.set_title('Zero-shot AUC by Prompt Template (NIH 8 classes, 2000 samples)')
plt.tight_layout()
plt.savefig('prompt_ablation.png', dpi=150)
plt.show()
print('Done.')